Mentre il Matcher trova pattern generici, l'EntityRuler trova pattern e li etichetta direttamente come Entità Nominate (NER). È essenziale per integrare la conoscenza basata su regole con il modello statistico.

In [1]:
import spacy

Il Sentencizer è un componente molto leggero della pipeline di spaCy che serve solo a dividere il testo in frasi (sentence segmentation) — senza usare il parser grammaticale.

In [2]:
# Ri-carichiamo il modello piccolo (sm) per dimostrare la Sentencizer
nlp_sm = spacy.load("it_core_news_sm", disable=["parser", "ner"])
# Disabilitiamo parser e ner per velocizzare e isolare l'effetto del Sentencizer

In [3]:
# Testo complesso con abbreviazioni che potrebbero confondere la segmentazione
testo_frasi = "Il Prof. Rossi è arrivato. Ercolano è in Campania."

sents è un generatore di frasi (sentences) presente in ogni oggetto Doc

In [ ]:
# Creiamo l'oggetto Doc senza Sentencizer
doc_no_sentencizer = nlp_sm(testo_frasi)
print("--- 7.1. Senza Sentencizer (Parser/Senter di default disabilitati) ---")
# Il Doc verrà trattato come un'unica frase
print(f"Numero di frasi: {len(list(doc_no_sentencizer.sents))}")

In [6]:
# Aggiungiamo il Sentencizer (basato su regole predefinite)
nlp_sm.add_pipe("sentencizer")
doc_with_sentencizer = nlp_sm(testo_frasi)

In [8]:
print("\n--- Con Sentencizer ---")
# Ora il Sentencizer dovrebbe aver identificato correttamente le due frasi
frasi = list(doc_with_sentencizer.sents)
print(f"Numero di frasi: {len(frasi)}")
print(f"Frase 1: {frasi[0].text}")
print(f"Frase 2: {frasi[1].text}")
print(f"Frase 3: {frasi[2].text}")


--- Con Sentencizer ---
Numero di frasi: 3
Frase 1: Il Prof.
Frase 2: Rossi è arrivato.
Frase 3: Ercolano è in Campania.


In [9]:
from spacy.pipeline import EntityRuler

In [10]:
# Inizializza il modello (ripristiniamo la pipeline completa)
nlp_ruler = spacy.load("it_core_news_sm")

In [11]:
# Crea l'EntityRuler
ruler = EntityRuler(nlp_ruler)

In [12]:
# Definizione dei pattern (usiamo un nome di vulcano non standard per dimostrare)
patterns = [
    # Pattern basato su testo esatto
    {"label": "VULCANO", "pattern": "Fossa di Vulcano"},
    # Pattern basato su token e POS (esempio: Adjective + Geopolitical Entity)
    {"label": "AREA_GEOLOGICA", "pattern": [{"LOWER": "campi"}, {"LOWER": "flegrei"}]}
]

In [13]:
# Aggiunge i pattern al Ruler
ruler.add_patterns(patterns)

In [14]:
# Aggiunge l'EntityRuler alla pipeline. Deve essere eseguito 
# PRIMA del NER statistico!
# Questo perché le entità trovate da EntityRuler sono 
# "gold" e aiutano il modello statistico.
nlp_ruler.add_pipe("entity_ruler", before="ner")

In [15]:
testo_ruler = "I Campi Flegrei sono pericolosi. La Fossa di Vulcano emana gas."
doc_ruler = nlp_ruler(testo_ruler)

c:\Users\Roman\AppData\Local\Programs\Python\Python313\Lib\site-packages\spacy\pipeline\entityruler.py:366: UserWarning: [W036] The component 'entity_ruler' does not have any patterns defined.
  warnings.warn(Warnings.W036.format(name=self.name))


In [16]:
print("\n--- 7.2. EntityRuler ---")
for ent in doc_ruler.ents:
    # Si nota che le entità trovate dal Ruler hanno le label personalizzate
    print(f"Entità: {ent.text:<20} | Tipo: {ent.label_}")


--- 7.2. EntityRuler ---
Entità: Campi Flegrei        | Tipo: LOC
Entità: Fossa di Vulcano     | Tipo: LOC
